In this Notebook we will train several baseline models. These models should be well trained to have high accuray, but only need to be trained once. 

In [1]:
from sparseDPD import DataManager, PGJANET_NeuralNetwork, Volterra, PNTDNN_NeuralNetwork

# Same dataset/training style as refactored_notebook.ipynb
openDPD_folder = 'OpenDPD_datasets/DPA_160MHz'

#40,000 points
openDPD_dataManager = DataManager(
    num_test_points=98304,
    num_training_points=294912,
    num_validaiton_points=98304,
    openDPD_test_input_file=f'{openDPD_folder}/test_input.csv',
    openDPD_test_output_file=f'{openDPD_folder}/test_output.csv',
    openDPD_training_input_file=f'{openDPD_folder}/train_input.csv',
    openDPD_training_output_file=f'{openDPD_folder}/train_output.csv',
    openDPD_validation_input_file=f'{openDPD_folder}/val_input.csv',
    openDPD_validation_output_file=f'{openDPD_folder}/val_output.csv'
)

# Print Volterra Performance
volterra_forward_model = Volterra(num_nl_orders=3, num_memory_levels=3, dataset=openDPD_dataManager.training_dataset)

print(f" Volterra NMSE: {volterra_forward_model.calculate_volterra_nmse(openDPD_dataManager.test_dataset)} dB")

 Volterra NMSE: -35.03138313219892 dB


In [2]:
# Train baseline PGJANET with optimized settings

model_path = 'pgjanet_open.pt'

PGJANET_forward_nn = PGJANET_NeuralNetwork(
    num_memory_levels=50,
    model_type='PGJANETNetwork',
    forward_model=True,
    seq_stride=1,
    batch_size=512,  # Increased from 32 for better GPU utilization
    hidden_size=32
)

train_losses_fwd, valid_losses_fwd, best_epoch_fwd = PGJANET_forward_nn.get_best_model(
    num_epochs=200,
    training_dataset=openDPD_dataManager.training_dataset,
    validation_dataset=openDPD_dataManager.validation_dataset,
    learning_rate=1e-2,
    target_nmse = -42
)

fwd_nmse = PGJANET_forward_nn.calculate_forward_nmse(openDPD_dataManager.test_dataset)
print(f"Trained PGJANET NMSE: {fwd_nmse:.2f} dB")
print(f"Best epoch: {best_epoch_fwd}")
    
PGJANET_forward_nn.write_nn_to_file(model_path)
print(f"Saved trained model to: {model_path}")


Using cuda device
Epoch  10/200  Loss=3.0481e+01  Valid Loss=8.8595e+00  LR=1.00e-02  NMSE=-36.7874 dB
Epoch  20/200  Loss=2.3284e+01  Valid Loss=8.0149e+00  LR=1.00e-02  NMSE=-37.2225 dB
Epoch  30/200  Loss=2.1368e+01  Valid Loss=9.0841e+00  LR=1.00e-02  NMSE=-36.6787 dB
Epoch  40/200  Loss=1.1250e+05  Valid Loss=4.0670e+04  LR=5.00e-03  NMSE=-0.1664 dB
Epoch  50/200  Loss=1.0999e+05  Valid Loss=3.9865e+04  LR=2.50e-03  NMSE=-0.2719 dB
Epoch  60/200  Loss=1.1589e+05  Valid Loss=4.1812e+04  LR=1.25e-03  NMSE=-0.0441 dB
Epoch  70/200  Loss=1.1577e+05  Valid Loss=4.1776e+04  LR=6.25e-04  NMSE=-0.0515 dB
Epoch  80/200  Loss=1.1572e+05  Valid Loss=4.1755e+04  LR=3.13e-04  NMSE=-0.0527 dB
Epoch  90/200  Loss=1.1575e+05  Valid Loss=4.1765e+04  LR=1.56e-04  NMSE=-0.0489 dB
Epoch 100/200  Loss=1.1579e+05  Valid Loss=4.1791e+04  LR=1.56e-04  NMSE=-0.0518 dB
Epoch 110/200  Loss=1.1577e+05  Valid Loss=4.1795e+04  LR=7.81e-05  NMSE=-0.0551 dB
Epoch 120/200  Loss=1.1572e+05  Valid Loss=4.1747e+04  

In [3]:
from sparseDPD import PNTDNN_NeuralNetwork

model_path = 'pntdnn_one_layer_open.pt'

PNTDNN_forward_nn = PNTDNN_NeuralNetwork(
    num_memory_levels=15,
    forward_model=True,
    model_type = 'OneLayerNetwork'
)

train_losses_fwd, valid_losses_fwd, best_epoch_fwd = PNTDNN_forward_nn.get_best_model(
    num_epochs=600,
    training_dataset=openDPD_dataManager.training_dataset,
    validation_dataset=openDPD_dataManager.validation_dataset,
    learning_rate=1e-3,
    target_nmse = -42
)

fwd_nmse = PNTDNN_forward_nn.calculate_forward_nmse(openDPD_dataManager.test_dataset)
print(f"Trained PNTDNN NMSE: {fwd_nmse:.2f} dB")
print(f"Best epoch: {best_epoch_fwd}")
    
PNTDNN_forward_nn.write_nn_to_file(model_path)
print(f"Saved trained model to: {model_path}")

Using cuda device
Epoch  10/600  Loss=1.7648e+01  Valid Loss=1.0115e+01  LR=1.00e-03  NMSE=-36.2141 dB
Epoch  20/600  Loss=1.4942e+01  Valid Loss=7.5806e+00  LR=1.00e-03  NMSE=-37.4666 dB
Epoch  30/600  Loss=1.4285e+01  Valid Loss=7.2736e+00  LR=1.00e-03  NMSE=-37.6461 dB
Epoch  40/600  Loss=1.3984e+01  Valid Loss=7.2943e+00  LR=1.00e-03  NMSE=-37.6338 dB
Epoch  50/600  Loss=1.3057e+01  Valid Loss=5.3508e+00  LR=5.00e-04  NMSE=-38.9794 dB
Epoch  60/600  Loss=1.2537e+01  Valid Loss=6.5905e+00  LR=2.50e-04  NMSE=-38.0744 dB
Epoch  70/600  Loss=1.2372e+01  Valid Loss=5.1033e+00  LR=1.25e-04  NMSE=-39.1851 dB
Epoch  80/600  Loss=1.2313e+01  Valid Loss=4.8699e+00  LR=6.25e-05  NMSE=-39.3884 dB
Epoch  90/600  Loss=1.2301e+01  Valid Loss=4.8660e+00  LR=6.25e-05  NMSE=-39.3919 dB
Epoch 100/600  Loss=1.2291e+01  Valid Loss=4.8615e+00  LR=6.25e-05  NMSE=-39.3959 dB
Epoch 110/600  Loss=1.2281e+01  Valid Loss=4.8575e+00  LR=6.25e-05  NMSE=-39.3995 dB
Epoch 120/600  Loss=1.2271e+01  Valid Loss=4.85

In [4]:
model_path = 'pntdnn_three_layer_open.pt'

PNTDNN_three_layer_forward_nn = PNTDNN_NeuralNetwork(
    num_memory_levels=15,
    forward_model=True,
    model_type = 'ThreeLayerNetwork'
)

train_losses_fwd, valid_losses_fwd, best_epoch_fwd = PNTDNN_three_layer_forward_nn.get_best_model(
    num_epochs=600,
    training_dataset=openDPD_dataManager.training_dataset,
    validation_dataset=openDPD_dataManager.validation_dataset,
    learning_rate=1e-3,
    target_nmse = -42
)

fwd_nmse = PNTDNN_three_layer_forward_nn.calculate_forward_nmse(openDPD_dataManager.test_dataset)
print(f"Trained PNTDNN NMSE: {fwd_nmse:.2f} dB")
print(f"Best epoch: {best_epoch_fwd}")
    
PNTDNN_three_layer_forward_nn.write_nn_to_file(model_path)
print(f"Saved trained model to: {model_path}")

Using cuda device
Epoch  10/600  Loss=1.8126e+01  Valid Loss=1.0041e+01  LR=1.00e-03  NMSE=-36.2459 dB
Epoch  20/600  Loss=1.4164e+01  Valid Loss=6.6148e+00  LR=5.00e-04  NMSE=-38.0585 dB
Epoch  30/600  Loss=1.3060e+01  Valid Loss=6.4212e+00  LR=2.50e-04  NMSE=-38.1875 dB
Epoch  40/600  Loss=1.2580e+01  Valid Loss=5.4976e+00  LR=1.25e-04  NMSE=-38.8619 dB
Epoch  50/600  Loss=1.2442e+01  Valid Loss=5.5639e+00  LR=1.25e-04  NMSE=-38.8098 dB
Epoch  60/600  Loss=1.2364e+01  Valid Loss=4.8161e+00  LR=6.25e-05  NMSE=-39.4367 dB
Epoch  70/600  Loss=1.2333e+01  Valid Loss=4.8046e+00  LR=6.25e-05  NMSE=-39.4471 dB
Epoch  80/600  Loss=1.2305e+01  Valid Loss=4.7940e+00  LR=6.25e-05  NMSE=-39.4567 dB
Epoch  90/600  Loss=1.2279e+01  Valid Loss=4.7844e+00  LR=6.25e-05  NMSE=-39.4653 dB
Epoch 100/600  Loss=1.2255e+01  Valid Loss=4.7761e+00  LR=6.25e-05  NMSE=-39.4729 dB
Epoch 110/600  Loss=1.2234e+01  Valid Loss=4.7678e+00  LR=6.25e-05  NMSE=-39.4805 dB
Epoch 120/600  Loss=1.2213e+01  Valid Loss=4.75

In [ ]:
model_path = 'pntdnn_multi_layer_open.pt'

PNTDNN_multi_layer_forward_nn = PNTDNN_NeuralNetwork(
    num_memory_levels=15,
    forward_model=True,
    model_type = 'MultiLayerNetwork'
)

train_losses_fwd, valid_losses_fwd, best_epoch_fwd = PNTDNN_multi_layer_forward_nn.get_best_model(
    num_epochs=600,
    training_dataset=openDPD_dataManager.training_dataset,
    validation_dataset=openDPD_dataManager.validation_dataset,
    learning_rate=1e-3,
    target_nmse = -42
)

fwd_nmse = PNTDNN_multi_layer_forward_nn.calculate_forward_nmse(openDPD_dataManager.test_dataset)
print(f"Trained PNTDNN NMSE: {fwd_nmse:.2f} dB")
print(f"Best epoch: {best_epoch_fwd}")
    
PNTDNN_multi_layer_forward_nn.write_nn_to_file(model_path)
print(f"Saved trained model to: {model_path}")

Using cuda device
Epoch  10/600  Loss=1.0094e+01  Valid Loss=1.5326e+00  LR=1.00e-03  NMSE=-20.6202 dB
Epoch  20/600  Loss=1.8905e+00  Valid Loss=2.8457e-01  LR=1.00e-03  NMSE=-27.9326 dB
Epoch  30/600  Loss=9.2576e-01  Valid Loss=1.6420e-01  LR=1.00e-03  NMSE=-30.3208 dB
Epoch  40/600  Loss=5.1790e-01  Valid Loss=9.7998e-02  LR=1.00e-03  NMSE=-32.5622 dB
Epoch  50/600  Loss=3.8028e-01  Valid Loss=6.5338e-02  LR=1.00e-03  NMSE=-34.3227 dB
Epoch  60/600  Loss=2.9803e-01  Valid Loss=5.0142e-02  LR=1.00e-03  NMSE=-35.4724 dB
Epoch  70/600  Loss=2.4636e-01  Valid Loss=4.1945e-02  LR=1.00e-03  NMSE=-36.2476 dB
Epoch  80/600  Loss=2.8828e-01  Valid Loss=3.9637e-02  LR=1.00e-03  NMSE=-36.4934 dB
Epoch  90/600  Loss=2.5396e-01  Valid Loss=3.5260e-02  LR=1.00e-03  NMSE=-37.0016 dB
Epoch 100/600  Loss=3.1546e-01  Valid Loss=3.1547e-02  LR=1.00e-03  NMSE=-37.4849 dB
Epoch 110/600  Loss=7.7461e-01  Valid Loss=5.7536e-02  LR=1.00e-03  NMSE=-34.8750 dB
Epoch 120/600  Loss=1.1158e-01  Valid Loss=2.33

In [ ]:
from sparseDPD import ARVTDNN_NeuralNetwork

model_path = 'arvtdnn_one_layer_open.pt'

ARVTDNN_one_layer_forward_nn = ARVTDNN_NeuralNetwork(
    num_memory_levels=15,
    forward_model=True,
    model_type = 'OneLayerNetwork'
)

train_losses_fwd, valid_losses_fwd, best_epoch_fwd = ARVTDNN_one_layer_forward_nn.get_best_model(
    num_epochs=600,
    training_dataset=openDPD_dataManager.training_dataset,
    validation_dataset=openDPD_dataManager.validation_dataset,
    learning_rate=1e-3,
    target_nmse = -42
)

fwd_nmse = ARVTDNN_one_layer_forward_nn.calculate_forward_nmse(openDPD_dataManager.test_dataset)
print(f"Trained ARVTDNN NMSE: {fwd_nmse:.2f} dB")
print(f"Best epoch: {best_epoch_fwd}")
    
ARVTDNN_one_layer_forward_nn.write_nn_to_file(model_path)
print(f"Saved trained model to: {model_path}")

Using cuda device
Epoch  10/600  Loss=6.8254e+00  Valid Loss=1.0823e+00  LR=1.00e-03  NMSE=-22.1311 dB
Epoch  20/600  Loss=2.3267e+00  Valid Loss=5.3051e-01  LR=1.00e-03  NMSE=-25.2275 dB
Epoch  30/600  Loss=1.5461e+00  Valid Loss=3.3400e-01  LR=1.00e-03  NMSE=-27.2370 dB
Epoch  40/600  Loss=1.0753e+00  Valid Loss=2.6560e-01  LR=1.00e-03  NMSE=-28.2321 dB
Epoch  50/600  Loss=6.5426e-01  Valid Loss=2.1986e-01  LR=1.00e-03  NMSE=-29.0529 dB
Epoch  60/600  Loss=6.1443e-01  Valid Loss=9.8553e-02  LR=1.00e-03  NMSE=-32.5377 dB
Epoch  70/600  Loss=5.8828e-01  Valid Loss=6.7204e-02  LR=1.00e-03  NMSE=-34.2004 dB
Epoch  80/600  Loss=3.0845e-01  Valid Loss=7.0662e-02  LR=5.00e-04  NMSE=-33.9825 dB
Epoch  90/600  Loss=1.9145e-01  Valid Loss=4.2409e-02  LR=2.50e-04  NMSE=-36.1998 dB
Epoch 100/600  Loss=1.7785e-01  Valid Loss=4.0484e-02  LR=2.50e-04  NMSE=-36.4016 dB
Epoch 110/600  Loss=1.7611e-01  Valid Loss=3.9379e-02  LR=2.50e-04  NMSE=-36.5217 dB
Epoch 120/600  Loss=1.8296e-01  Valid Loss=4.07